# Pelatihan Klasifikasi Aktivitas dengan ESP32

Notebook ini mengevaluasi model klasifikasi diam, jalan, dan lari. File contoh
`dataset_aktivitas_primer.csv` berisi sinyal sintetis dari
`generate_handheld_dataset.py`. Untuk menilai kinerja pada rekaman nyata, ubah
`DATASET` ke CSV lokal dan gunakan ID yang konsisten untuk setiap peserta.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.tree import DecisionTreeClassifier

from har_pipeline import CLASSES, CLASS_IDS, evaluate_folds, make_mlp, read_recordings, subject_folds

DATASET = Path("dataset_aktivitas_primer.csv")
data = read_recordings(DATASET)
print(f"{len(data):,} baris, {data['subjek'].nunique()} subjek")
display(data["label"].value_counts().reindex(CLASSES).rename("Sampel"))

## Pembagian subjek dan ekstraksi fitur

Tiap fold menahan seluruh rekaman satu subjek. Jendela dibentuk secara terpisah
di set latih dan uji dengan panjang 64 sampel dan langkah 32. Rata-rata,
simpangan baku, minimum, maksimum, dan RMS dihitung untuk setiap sumbu
akselerometer dan magnitudo. Pipeline MLP melakukan fit StandardScaler pada
set latih di setiap fold.

In [ ]:
folds = list(subject_folds(data))
models = {
    "MLP": make_mlp,
    "Decision tree": lambda: DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random forest": lambda: RandomForestClassifier(n_estimators=30, max_depth=5, random_state=42),
}
results = {name: evaluate_folds(folds, build) for name, build in models.items()}
display(pd.DataFrame([
    {
        "Model": name,
        "Akurasi": accuracy_score(truth, predicted),
        "Macro F1": f1_score(truth, predicted, labels=CLASS_IDS, average="macro", zero_division=0),
    }
    for name, (_, truth, predicted) in results.items()
]).set_index("Model"))

## Evaluasi MLP

Prediksi di bawah berasal dari fold yang menahan subjek bersangkutan. Ketiga
arsitektur memakai fold yang sama sebagai perbandingan eksploratif. Jika
hasil ini dipakai untuk memilih model, gunakan rekaman terpisah dari subjek
baru untuk pengujian akhir.

In [ ]:
scores, truth, predicted = results["MLP"]
fold_scores = pd.DataFrame(
    [score[1:] for score in scores], columns=["Jendela latih", "Jendela uji", "Akurasi"]
)
fold_scores.index = pd.RangeIndex(1, len(fold_scores) + 1, name="Fold")
display(fold_scores)
print(classification_report(truth, predicted, labels=CLASS_IDS,
                            target_names=CLASSES, zero_division=0))
ax = sns.heatmap(confusion_matrix(truth, predicted, labels=CLASS_IDS),
                 annot=True, fmt="d", cmap="Blues", cbar=False,
                 xticklabels=CLASSES, yticklabels=CLASSES)
ax.set(xlabel="Prediksi", ylabel="Label sebenarnya", title="Matriks Konfusi MLP per Subjek")
plt.show()

## Ekspor model firmware

Jalankan `python train_and_export.py` untuk artefak demo atau
`python train_and_export.py --dataset data_local/rekaman_aktivitas.csv` untuk
rekaman lokal. Setelah evaluasi, skrip melatih ulang MLP pada seluruh input
dan mengekspor scaler, bobot, bias, dan fungsi inferensi ke `model_weights.h`.
Artefak dari CSV lokal tersimpan di `data_local/training/` secara bawaan.
Metrik validasi berasal dari model tiap fold.